# Mini Projekt SSNE 6

## Instalacja bibliotek

Przez pewien bug biblioteki `datasets` konieczny jest jej update do najnowszej wersji.

In [15]:
!pip install datasets -U

## Zamontowanie Dysku Google

In [16]:
import os
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [17]:
gdrive = "/content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_6/"

In [18]:
!ls /content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_6

 best_kod65.ipynb
 hate_test_data.txt
 hate_train.csv
 kod.ipynb
 pred_acc0.9291_20260609_195049.csv
'pred_acc=0.9318_f1=0.6442_20260613_213719.csv'
'pred_acc=0.9318_f1=0.6442_20260613_214201.csv'
'pred_acc=0.9392_f1=0.6369_20260610_185426.csv'
'pred_acc=0.9418_f1=0.6528_20260612_140539.csv'
 pred.csv


## Funkcje pomocnicze

In [19]:
from datetime import datetime
def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

## Importy bibliotek

In [20]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

## Konfiguracja

In [21]:
MODEL_NAME = "allegro/herbert-base-cased"
TRAIN_DATA_FILE = f"{gdrive}hate_train.csv"
RNG = 42
N_SPLITS = 5
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
NUM_EPOCHS = 6
EVAL_PER_EPOCH = 5
PATIENCE = 5
COMPARE_METRIC = "f1"  # "f1" | "accuracy"

## Ładowanie i wstępne procesowanie danych

Zauważyliśmy, że w zbiorze danych występują retweety będące kopiami istniejących w zbiorze wypowiedzi zaczynające się od "RT". Postanowiliśmy je usunąć, aby ujednolicić zbiór danych.  
  
Zamieniliśmy również występujące w tekście "@anonymized_account" na krótsze "@USER", aby uprościć reprezentację tokenową tekstu.

In [22]:
df = pd.read_csv(TRAIN_DATA_FILE)

df["sentence"] = (
    df["sentence"]
    .astype(str)
    .str.replace(r"^RT\s+", "", regex=True)
)

df["sentence"] = (
    df["sentence"]
    .str.replace(r"@\w+", "@USER", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

texts = df["sentence"].astype(str).tolist()
labels = df["label"].tolist()

## Framework Hugging Face `Trainer`

### Tokenizator

In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

### Funkcja obliczająca metryki

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

### `Trainer` z ważoną funkcją straty

Możliwe, że użycie `WeightedRandomSampler` zamiast tego dałoby stabilniejsze uczenie.

In [25]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

## Trening modelu ze stratyfikowaną $k$-krotną walidacją krzyżową

In [ ]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RNG
)

fold_results = []
best_metrics = {"accuracy": -torch.inf, "f1": -torch.inf}
best_model_name = None

for fold, (train_idx, val_idx) in enumerate(
    skf.split(texts, labels),
    start=1
):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_df["label"]),
        y=train_df["label"]
    )
    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float
    )
    print("Class weights:", class_weights)

    train_dataset = Dataset.from_pandas(
        train_df[["sentence", "label"]]
    )
    val_dataset = Dataset.from_pandas(
        val_df[["sentence", "label"]]
    )
    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True
    )
    val_dataset = val_dataset.map(
        tokenize_function,
        batched=True
    )
    train_dataset.set_format(
        type="torch",
        columns=[
            "input_ids",
            "attention_mask",
            "label"
        ]
    )
    val_dataset.set_format(
        type="torch",
        columns=[
            "input_ids",
            "attention_mask",
            "label"
        ]
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    steps_per_epoch = len(train_dataset) // BATCH_SIZE
    eval_steps = max(1, steps_per_epoch // EVAL_PER_EPOCH)
    output_dir = f"/content/fold_{fold}_trainer"

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,

        num_train_epochs=NUM_EPOCHS,

        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        eval_steps=eval_steps,
        save_steps=eval_steps,
        logging_steps=eval_steps,

        load_best_model_at_end=True,
        metric_for_best_model=COMPARE_METRIC,
        greater_is_better=True,

        weight_decay=0.01,

        fp16=torch.cuda.is_available(),
        save_total_limit=1,
        save_only_model=True,
        report_to="none"
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,

        train_dataset=train_dataset,
        eval_dataset=val_dataset,

        compute_metrics=compute_metrics,

        class_weights=class_weights,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=PATIENCE
            )
        ]
    )

    trainer.train()

    metrics = trainer.evaluate()
    fold_accuracy = metrics["accuracy"] = metrics["eval_accuracy"]
    fold_f1 = metrics["f1"] = metrics["eval_f1"]

    print(f"\nFold accuracy: {fold_accuracy:.4f} Fold F1: {fold_f1:.4f}")

    fold_results.append(metrics)
    save_name = f"acc={fold_accuracy:.4f}_f1={fold_f1:.4f}_{timestamp()}"
    trainer.save_model(f"/content/{save_name}")

    if metrics[COMPARE_METRIC] > best_metrics[COMPARE_METRIC]:
        best_metrics = metrics
        best_model_name = save_name

print("\n" + "="*60)
print("CROSS VALIDATION RESULTS")
print("="*60)

accuracies, f1s = list(zip(*[(result["accuracy"], result["f1"]) for result in fold_results]))

print(f"Mean accuracy: {np.mean(accuracies):.4f}")
print(f"Std accuracy:  {np.std(accuracies):.4f}")
print(f"Mean F1: {np.mean(f1s):.4f}")
print(f"Std F1:  {np.std(f1s):.4f}")

print(f"\nBest fold {COMPARE_METRIC}: {best_metrics[COMPARE_METRIC]:.4f}")
print(f"Best model name: {best_model_name}")



FOLD 1
Class weights: tensor([0.5462, 5.9059])


Map:   0%|          | 0/8032 [00:00<?, ? examples/s]

Map:   0%|          | 0/2009 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.072940,0.844299,2700,0.939741,0.664430,0.582353,0.620690



Fold accuracy: 0.9397 Fold F1: 0.6207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


CROSS VALIDATION RESULTS
Mean accuracy: 0.9305
Std accuracy:  0.0115
Mean F1: 0.6020
Std F1:  0.0441

Best fold f1: 0.6746
Best model name: acc=0.9452_f1=0.6746_20260613_220057


## Zapis parametrów modelu do pliku (opcjonalne)

In [ ]:
# models take up over 400MB, use at your own discretion
# import shutil
# shutil.copytree(f"/content/{best_model_name}", f"{gdrive}{best_model_name}")

## Klasyfikacja na zbiorze testowym

Tę komórkę można uruchomić nawet jeśli poprzednie nie zostały wykonane.  
Należy w takim wypadku podać nazwę folderu z zapisanymi parametrami modelu jako `SAVED_NAME`.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import re

gdrive = "/content/drive/MyDrive/sroda_FijalkowskiFilip_ZukowskaRadoslawa/mini_projekt_6/"

MODEL_NAME = "allegro/herbert-base-cased"
TEST_DATA_FILE = f"{gdrive}hate_test_data.txt"
try:
    SAVED_NAME = best_model_name
    MODEL_PATH = f"/content/{SAVED_NAME}"
except NameError:
    SAVED_NAME = "model"  # insert model save dir name here if you're just evaluating a saved model
    MODEL_PATH = f"{gdrive}{SAVED_NAME}"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

with open(TEST_DATA_FILE, "r", encoding="utf-8") as f:
    texts = [
        re.sub(r"\s+", " ",
               re.sub(r"@\w+", "@USER",
                      re.sub(r"^RT\s+", "", line.rstrip("\n"))))
        .strip()
        for line in f
    ]

predictions = []

batch_size = 32

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    preds = torch.argmax(logits, dim=1)

    predictions.extend(preds.cpu().numpy())

csv_name = f"pred_{SAVED_NAME}.csv"
pd.DataFrame(predictions).to_csv(
    f"{gdrive}{csv_name}",
    index=False,
    header=False
)

print(f"Saved {len(predictions)} predictions to {csv_name}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Saved 1000 predictions to pred_acc=0.9452_f1=0.6746_20260613_220057.csv


Sprawdziliśmy również inne możliwe warianty:
- trening bez ważonych klas (zwykle uzyskiwane $\text{F1}=0$)
- modyfikacja wag klas (np. $\log(1+N/n_c)$) (gorsze F1, lepsze Accuracy)  
  
Ostatecznie przyjęliśmy F1 jako najbardziej miarodajną metrykę jakości modelu (ze względu na nierówny rozkład klas predykcja samych zer dawałaby np. 90% Accuracy ale zerowe F1).